# ЛР 3. Идентичность, соли и таутомеры

**Сущность:** Построить проверяемую таблицу raw→parent и сохранить смысл химических различий.

**Данные:** molecules.csv: 12 исходных записей, два ошибочных ввода, соль, ковалентный Cl и стереопара.

**Workflow:** ID/raw → парсинг → явная политика → parent → журнал и контрпримеры.

Перед запуском прочитайте [guideline](../guides/lab03.md) и [теорию](../../Module_theory/Module%203_compound%20and%20reaction-representation.md).

## 1. Прогноз

Запишите ожидаемое поведение и отрицательный контроль. Какое изменение входа должно изменить результат, а какое — сохранить?

In [1]:
from pathlib import Path
import sys, json, math, csv, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# WHY: relative discovery makes the notebook portable across machines.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Labs" / "chemistry.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите notebook внутри репозитория курса")
sys.path.insert(0, str(ROOT / "Labs"))
DATA = ROOT / "Labs" / "datasets"
OUT = ROOT / "Labs" / "results" / "lab03"
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()


## 2. Разбор и исполнение

1. Спрогнозируйте пары эквивалентных графов до запуска.
2. Обработайте все строки, включая ошибки, не удаляя ID.
3. Проверьте отдельно ковалентный Cl, стереопару и объединение таутомеров по переключателю.

Ожидаемое: 12 строк на выходе; M08/M09 invalid; M01=M02 и M03=M04 по parent; M10≠M11.

In [2]:
from chemistry import screen, standardize, parse
raw = pd.read_csv(DATA/"molecules.csv", keep_default_na=False)
records = [{"id": row.id, **screen(row.smiles)} for row in raw.itertuples()]
report = pd.DataFrame(records); display(report)
report.to_csv(OUT/"curated.csv", index=False)
by_id = {row["id"]:row for row in records}
assert len(records)==12 and sum(row["status"]=="invalid" for row in records)==2
assert by_id["M01"]["parent_smiles"]==by_id["M02"]["parent_smiles"]
assert by_id["M03"]["parent_smiles"]==by_id["M04"]["parent_smiles"]
assert "Cl" in by_id["M05"]["parent_smiles"]
assert by_id["M10"]["parent_smiles"]!=by_id["M11"]["parent_smiles"]
pair=["O=c1cccc[nH]1","Oc1ccccn1"]
assert standardize(pair[0]).parent_smiles != standardize(pair[1]).parent_smiles
assert standardize(pair[0],True).parent_smiles == standardize(pair[1],True).parent_smiles
artifact={"rows":len(records),"invalid":2,"policy":by_id["M01"]["policy"]}


[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Running LargestFragmentChooser
[11:58:35] Running Uncharger
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Running LargestFragmentChooser
[11:58:35] Running Uncharger
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Norma

[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Running LargestFragmentChooser
[11:58:35] Running Uncharger
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Running LargestFragmentChooser
[11:58:35] Running Uncharger


,id,raw_smiles,parent_smiles,policy,MolWt,MolLogP,TPSA,HBD,HBA,ro5_violations,alerts,status,reason
0,M01,CCO,CCO,teaching-parent-v1;neutralize;tautomer=False,46.069,-0.0014,20.23,1.0,1.0,0.0,[],eligible,No listed alert; activity remains unknown
1,M02,OCC,CCO,teaching-parent-v1;neutralize;tautomer=False,46.069,-0.0014,20.23,1.0,1.0,0.0,[],eligible,No listed alert; activity remains unknown
2,M03,CC(=O)[O-].[Na+],CC(=O)O,teaching-parent-v1;neutralize;tautomer=False,60.052,0.0909,37.30,1.0,1.0,0.0,[],eligible,No listed alert; activity remains unknown
3,M04,CC(=O)O,CC(=O)O,teaching-parent-v1;neutralize;tautomer=False,60.052,0.0909,37.30,1.0,1.0,0.0,[],eligible,No listed alert; activity remains unknown
4,M05,Clc1ccccc1,Clc1ccccc1,teaching-parent-v1;neutralize;tautomer=False,112.559,2.3400,0.00,0.0,0.0,0.0,[],eligible,No listed alert; activity remains unknown
5,M06,Oc1ccccc1O,Oc1ccccc1O,teaching-parent-v1;neutralize;tautomer=False,110.112,1.0978,40.46,2.0,2.0,0.0,[catechol_A(92)],review,PAINS:catechol_A(92)
6,M07,c1ccccc1,c1ccccc1,teaching-parent-v1;neutralize;tautomer=False,78.114,1.6866,0.00,0.0,0.0,0.0,[],eligible,No listed alert; activity remains unknown
7,M08,not_a_smiles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,invalid,Invalid SMILES
8,M09,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,invalid,SMILES must be a nonempty string
9,M10,C[C@H](O)C(=O)O,C[C@H](O)C(=O)O,teaching-parent-v1;neutralize;tautomer=False,90.078,-0.5482,57.53,2.0,2.0,0.0,[],eligible,No listed alert; activity remains unknown


[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Running LargestFragmentChooser
[11:58:35] Running Uncharger
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Running LargestFragmentChooser
[11:58:35] Running Uncharger
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Normalizer
[11:58:35] Initializing MetalDisconnector
[11:58:35] Running MetalDisconnector
[11:58:35] Initializing Normalizer
[11:58:35] Running Norma

## 3. Проверяемая запись запуска

Benchmark фиксирует этот запуск. Время включает расчёт и создание артефактов, но не установку среды; это не сравнительный рейтинг оборудования.

In [3]:
elapsed = time.perf_counter()-started
run_record = {"lab": 3, "calculation_wall_seconds": elapsed, "result": artifact}
(OUT/"benchmark.json").write_text(json.dumps(run_record,ensure_ascii=False,indent=2),encoding="utf-8")
print("Запуск сохранён:",OUT/"benchmark.json")


Запуск сохранён: C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\practice-audit-issue1-20261117\Labs\results\lab03\benchmark.json


## 4. Самостоятельная тренировка

1. Добавьте в копию датасета бромбензол; сохранится ли Br?
2. Поменяйте порядок компонентов соли; сравните parent.
3. Объясните, почему одинаковый parent не разрешает смешать растворимость соли и кислоты.

[Ответы после попытки](../solutions/lab03.md). Вернитесь к первому заданию через неделю.

## 5. Ваш научный вывод

1. Какой результат получен?
2. Какой вывод из него пока не следует?
3. Какой следующий эксперимент или контроль нужен?

Сохраните свой вариант notebook отдельно; исходный пример должен остаться воспроизводимым.